In [ ]:
# ===== v38 prior-PF 単体提出: 設定 + データ/成果物発見 =====
import os, glob, json, time, math
from pathlib import Path
import numpy as np, pandas as pd
import torch

def _find_data():
    for p in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction"]:
        if Path(p, "sample_submission.csv").exists(): return Path(p)
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    for h in hits:
        if Path(h).parent.joinpath("test").exists(): return Path(h).parent
    return Path(r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction")
def _find_file(name, local):
    if Path(local).exists(): return Path(local)
    root = Path("/kaggle/input")
    if root.exists():
        hits = list(root.glob(f"**/{name}"))
        if hits: return hits[0]
    return None

DATA = _find_data(); TEST_DIR = DATA / "test"
V38_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v38\artifacts_v38")
PACK = json.loads(_find_file("v38_submit_pack.json", V38_ART / "v38_submit_pack.json").read_text())
FZ = np.load(_find_file("struct_field_v38.npz", V38_ART / "struct_field_v38.npz"))
BP = PACK["params"]; ANCHOR_MODE = PACK["anchor_mode"]; ANCHOR_MULT = float(PACK["anchor_mult"])
BIAS_MULT = float(PACK["bias_mult"]); SEED = int(PACK["seed"]); N_SEEDS = int(PACK["n_seeds"])
WELL_CHUNK = int(PACK["well_chunk"])
FIELD = FZ["field"]; KAPPA = FZ["kappa"]; CENTERS = FZ["centers"]; SIGS = FZ["sigs"]
THETA0 = float(FZ["theta0"]); KSEG = int(FZ["K"]); KNN_K = int(FZ["knn_k"]); KNN_H = float(FZ["knn_h"])
GATE = float(FZ["gate"]); KBINS = FZ["kbins"]; SIG_MIN = float(FZ["sig_min"]); SIG_MAX = float(FZ["sig_max"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; DTYPE = torch.float32
print(f"DATA={DATA} | study={PACK['study']} mode={ANCHOR_MODE} mult={ANCHOR_MULT:g} "
      f"seed={SEED} n_seeds={N_SEEDS} | device={DEVICE} | field {FIELD.shape}")


In [ ]:
# ===== test構造prior(K16 dip場, 実行時構築)=====
def _seg(X, Y, s, n):
    edges = np.linspace(0, n, KSEG + 1)
    segid = np.clip(np.searchsorted(edges[1:], np.arange(1, n + 1.0), side="left"), 0, KSEG - 1)
    mid = np.empty((KSEG, 2)); proj = np.empty(KSEG)
    for j in range(KSEG):
        f0 = s + 1 + int(edges[j]); f1 = min(s + 1 + max(int(edges[j + 1]) - 1, int(edges[j])), len(X) - 1)
        az = np.degrees(np.arctan2(Y[f1] - Y[f0], X[f1] - X[f0])) % 360
        mid[j] = ((X[f0] + X[f1]) / 2, (Y[f0] + Y[f1]) / 2); proj[j] = np.cos(np.radians(az - THETA0))
    return segid, mid, proj
def _knn_full(mid):
    fx, fy, fD = FIELD[:, 0], FIELD[:, 1], FIELD[:, 2]
    Dh = np.empty(KSEG); dd = np.empty(KSEG)
    for j in range(KSEG):
        d2 = (fx - mid[j, 0]) ** 2 + (fy - mid[j, 1]) ** 2
        sel = np.argpartition(d2, KNN_K)[:KNN_K]
        w = np.exp(np.maximum(-d2[sel] / (2 * KNN_H ** 2), -700))
        Dh[j] = np.sum(w * fD[sel]) / np.sum(w); dd[j] = np.sqrt(np.median(d2[sel]))
    return Dh, dd
def struct_for(Xc, Yc, Zc, ti, s, n):
    segid, mid, proj = _seg(Xc, Yc, s, n); Dh, dd = _knn_full(mid)
    g = ndz = -np.diff(Zc)[s:]
    g = ndz + (Dh * proj)[segid]; g = np.where(np.abs(proj[segid]) < GATE, 0.0, g)
    kstep = KAPPA[np.digitize(dd, KBINS[1:-1])][segid]
    tvt = float(ti[s]) + np.cumsum(kstep * g)
    sig = np.clip(np.interp(dd[segid], CENTERS, SIGS), SIG_MIN, SIG_MAX)
    return tvt, sig


In [ ]:
# ===== v38 PFエンジン(test版: TVT不要)=====
def build_cache_test(wid):
    hw = pd.read_csv(TEST_DIR / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TEST_DIR / f"{wid}__typewell.csv").sort_values("TVT")
    ti = pd.to_numeric(hw["TVT_input"], errors="coerce").to_numpy(float)
    kn = np.where(np.isfinite(ti))[0]; evm = ~np.isfinite(ti); ev = np.where(evm)[0]
    if len(ev) == 0 or len(kn) < 10: return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    if len(tw_tvt) < 3: return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = pd.Series(hw["GR"].astype(float)).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    Z = hw["Z"].to_numpy(float); MD = hw["MD"].to_numpy(float)
    ls = float(ti[kn[-1]] + Z[kn[-1]])
    t = kn[-30:]; dt = np.diff(ti[t]); dz = np.diff(Z[t]); dm = np.diff(MD[t]); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    s = int(kn.max()); n = len(ev)
    try:
        anc, ancs = struct_for(hw["X"].to_numpy(float), hw["Y"].to_numpy(float), Z, ti, s, n)
        if len(anc) != n: raise ValueError
    except Exception:
        anc = np.zeros(n); ancs = np.full(n, 1e9)
    return dict(md=MD[ev], z=Z[ev], gr_ev_src=gr_filled, ev_pos=ev, kpos=kn,
                kn_tvtin=ti[kn], tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb, ls=ls, ir=ir,
                anc=np.asarray(anc, float), ancs=np.asarray(ancs, float),
                row_ids=[f"{wid}_{int(i)}" for i in ev], last_tvt=float(ti[kn[-1]]))

def _smoothv(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2 * r + 1, center=True, min_periods=1).mean().to_numpy(np.float32)

def make_inputs(c, P):
    tw_gr_pf = _smoothv(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    a, b = float(c["tw_tvt"].min()), float(c["tw_tvt"].max()); gg = np.arange(a, b + 0.2, 0.2)
    gg = np.interp(gg, c["tw_tvt"], tw_gr_pf); gmin, gst = a, 0.2
    gr_sm = _smoothv(c["gr_ev_src"], c["tw_fb"], P["hgr_smooth_r"])
    if len(c["kpos"]) < 20: gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64), gg=gg.astype(np.float64),
                gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], anc=c["anc"], ancs=c["ancs"])

def _pf_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, anc, ancs, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    AMUL = float(ANCHOR_MULT); WELLMEAN = (ANCHOR_MODE == "wellmean")
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts = torch.zeros((B, T), device=device, dtype=dtype)
    if WELLMEAN:
        vf = valid.to(dtype); sig_ok = (ancs < 1e8).to(dtype) * vf
        sb = (ancs * sig_ok).sum(1) / sig_ok.sum(1).clamp_min(1.0)
        sb = torch.where(sig_ok.sum(1) > 0, sb, torch.full_like(sb, 1e9)) * float(BIAS_MULT)
        sb2 = (sb * sb).clamp_min(1e-6)[:, None]
        Dm = torch.zeros((B, N), device=device, dtype=dtype); cnt = torch.zeros((B, 1), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype)
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero).clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if WELLMEAN:
            dev = v - anc[:, i][:, None]
            Dn = torch.where(am, (Dm * cnt + dev) / (cnt + 1.0), Dm)
            expo = (-0.5 * (Dn * Dn - Dm * Dm) / sb2).clamp(-60.0, 60.0)
            lka = torch.where(am, torch.exp(expo), torch.ones_like(w))
            Dm = Dn; cnt = cnt + act.to(dtype)[:, None]
        else:
            asig = (ancs[:, i][:, None] * AMUL).clamp_min(1e-6)
            da = (v - anc[:, i][:, None]) / asig
            lka = torch.where(da * da < 1200.0, torch.exp(-0.5 * da * da), zero).clamp_min(1e-300)
        w_new = torch.where(obs[:, None], w * lk * lka, w * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            if WELLMEAN: Dm = torch.where(nm, torch.gather(Dm, 1, idx), Dm)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
        pts[:, i] = (w * (pos - zi)).sum(1); pm = torch.where(act, cur, pm)
    return pts, log_lik

def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
        anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
        if Tn < Tmax: anc[b, Tn:] = anc[b, Tn - 1]
        ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs))

def run_pf(inps):
    N = int(BP["n_particles"]); S = N_SEEDS; W = len(inps); pad = _pad(inps, DEVICE)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=DEVICE); gen.manual_seed(SEED)
    pts, ll = _pf_core(rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                       pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                       pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                       rep(pad["anc"]), rep(pad["ancs"]), BP, N, DEVICE, gen)
    pts = pts.view(W, S, -1).double().cpu().numpy(); ll = ll.view(W, S).cpu().numpy()
    outs = []
    for k in range(W):
        T = len(inps[k]["md"]); pr = pts[k, :, :T]; llw = ll[k]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lkv = np.where(np.isfinite(llw), llw - mx, -np.inf)
            ww = np.exp(lkv / max(BP["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
        else:
            ww = np.full(S, 1.0 / S)
        outs.append((ww[:, None] * pr).sum(0))
    return outs


In [ ]:
# ===== 実行: test全坑井をチャンクでPF → フォールバック付きで予測収集 =====
t0 = time.time()
wids = sorted({Path(p).stem.replace("__horizontal_well", "")
               for p in glob.glob(str(TEST_DIR / "*__horizontal_well.csv"))})
caches = []
for w in wids:
    try:
        c = build_cache_test(w)
    except Exception as _e:
        c = None; print("cache skip:", w, type(_e).__name__)
    if c is not None: caches.append((w, c))
print(f"wells: {len(caches)}/{len(wids)}  (struct錨あり {sum(1 for _,c in caches if (c['ancs']<1e8).any())})")

pred_ids, pred_tvt = [], []
for i0 in range(0, len(caches), WELL_CHUNK):
    sub = caches[i0:i0 + WELL_CHUNK]
    try:
        inps = [make_inputs(c, BP) for _, c in sub]
        outs = run_pf(inps)
    except Exception as _e:
        outs = None; print(f"chunk {i0} PF failed: {type(_e).__name__} {str(_e)[:100]} -> struct/flatフォールバック")
    for k, (w, c) in enumerate(sub):
        if outs is not None and np.isfinite(outs[k]).all():
            p = outs[k]
        elif (c["ancs"] < 1e8).any():
            p = c["anc"]                                   # フォールバック1: 構造prior
        else:
            p = np.full(len(c["md"]), c["last_tvt"])       # フォールバック2: flat
        pred_ids += c["row_ids"]; pred_tvt.append(p.astype(np.float64))
    print(f"  chunk {i0//WELL_CHUNK+1}/{math.ceil(len(caches)/WELL_CHUNK)} done ({time.time()-t0:.0f}s)", flush=True)
pred_tvt = np.concatenate(pred_tvt)
pred_df = pd.DataFrame({"id": pred_ids, "tvt": pred_tvt})
print(f"predicted rows: {len(pred_df)}  ({time.time()-t0:.0f}s)")


In [ ]:
# ===== sample_submission 整列 → submission.csv =====
ss = pd.read_csv(DATA / "sample_submission.csv")
sub = ss[["id"]].merge(pred_df, on="id", how="left")
fill = float(np.nanmedian(pred_df["tvt"])) if len(pred_df) else 0.0
n_missing = int(sub["tvt"].isna().sum()); sub["tvt"] = sub["tvt"].fillna(fill)
sub.to_csv("submission.csv", index=False)
print("submission:", sub.shape, "| missing filled:", n_missing)
sub.head()
